# Cattle muzzle dataset: manifest, near-duplicate audit and identity-disjoint splits

This notebook is the first milestone of the muzzle-biometrics project. It loads the Hugging Face dataset, then:

1. builds a **manifest** (cow ID, image ID, camera/source prefix, size, quality flags),
2. audits **near-duplicates** (the leakage risk the brief warns about),
3. makes a **locked identity-disjoint split** (train / dev / test cows),
4. builds **enrolment / query** partitions for the dev and test cows,
5. generates **genuine and impostor pairs** and checks them for leakage,
6. saves `manifest.csv`, `splits.json`, `pairs_dev.csv`, `pairs_test.csv` and a draft `protocol.md` into `outputs/`.

**Resolution warning.** The Hugging Face copy is downscaled to at most 256 px on the long side. That is fine for auditing and splitting, but too small for bead/groove landmarks. For the landmark extractor, download the full-resolution Zenodo originals (10.5281/zenodo.6324361, CC BY 4.0), unzip them, set `LOCAL_DIR` below and rerun. The unzipped folder has one sub-folder per animal, which is exactly what `imagefolder` expects.

## 0. Setup

In [ ]:
%pip install -q datasets pillow pandas numpy matplotlib tqdm

In [ ]:
import io, json, re, hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image as PILImage
from tqdm.auto import tqdm
from datasets import load_dataset, Image as HFImage

HF_ID = "casperhuggingface/gaaibastu-muzzles"
LOCAL_DIR = None                  # e.g. "BeefCattle_Muzzle_database" to use the full-resolution Zenodo originals
EXCLUDE_CLASSES = {"perturbed"}   # the HF repo has a folder of demo images that is not a cow
META_CSV = None                   # optional local copy of data/muzzles.csv from the HF repo; fetched automatically when None

SEED = 42
SPLIT_FRACTIONS = {"train": 0.60, "dev": 0.20, "test": 0.20}   # fractions of COWS, not images
N_ENROLL = 3                      # enrolment images per held-out cow
GUARD_FRAMES = 2                  # a query must be more than this many frames from every enrolment frame
DUP_BITS = 16                     # dHash Hamming distance (of 256 bits) at or below which images count as near-duplicates
MIN_QUERY = 2                     # a held-out cow needs this many usable queries to take part in pairs
IMPOSTORS_PER_QUERY = 10          # same-source impostor pairs per query image
CROSS_SOURCE_IMPOSTORS_PER_QUERY = 3
HASH_SIZE = 16                    # dHash grid: 16 x 16 = 256 bits
MIN_SIDE = 96                     # short side below this (pixels) is flagged "small"

OUT = Path("outputs")
OUT.mkdir(exist_ok=True)

CONFIG = dict(
    dataset=(LOCAL_DIR or HF_ID), seed=SEED, split_fractions=SPLIT_FRACTIONS, n_enroll=N_ENROLL,
    guard_frames=GUARD_FRAMES, dup_bits=DUP_BITS, min_query=MIN_QUERY,
    impostors_per_query=IMPOSTORS_PER_QUERY, cross_source_impostors_per_query=CROSS_SOURCE_IMPOSTORS_PER_QUERY,
    hash_size=HASH_SIZE, min_side=MIN_SIDE,
)
CONFIG

## 1. Load the dataset and build the base manifest

The call below is the one-liner from the Hugging Face page. The repo also contains a `perturbed` folder of demo images, which `imagefolder` treats as an extra class, so it is dropped here.

In [ ]:
if LOCAL_DIR is None:
    ds = load_dataset(HF_ID)                                   # DatasetDict with a single "train" split
else:
    ds = load_dataset("imagefolder", data_dir=LOCAL_DIR)
print(ds)

train = ds["train"]
label_names = train.features["label"].names
print(f"{train.num_rows} rows, {len(label_names)} classes")
print(label_names[:5], "...", label_names[-3:])

In [ ]:
# File paths come from a second, non-decoding view of the same data (no pixels are loaded here).
paths = [rec["image"].get("path") for rec in train.cast_column("image", HFImage(decode=False))]
labels = train["label"]

# The HF repo also ships data/muzzles.csv (original file names and original pixel sizes). It is used to
# (a) recover the real file names if the loader exposes cache paths instead, and (b) record the original resolution.
meta = None
if LOCAL_DIR is None:
    try:
        meta_path = META_CSV
        if meta_path is None:
            from huggingface_hub import hf_hub_download
            meta_path = hf_hub_download(HF_ID, "data/muzzles.csv", repo_type="dataset")
        meta = pd.read_csv(meta_path)
        meta["file"] = meta["file"].astype(str).str.replace("\\", "/", regex=False).str.split("/").str[-1]
        print("metadata CSV columns:", list(meta.columns))
    except Exception as e:
        print("Could not read data/muzzles.csv (continuing without it):", repr(e)[:200])


def valid_name_fraction():
    ok = sum(1 for p, l in zip(paths, labels) if p and label_names[l] in p.replace("\\", "/").split("/")[-1])
    return ok / max(1, len(paths))


if valid_name_fraction() < 0.9:
    if meta is not None and {"cow", "file"} <= set(meta.columns):
        key = "order_in_cow" if "order_in_cow" in meta.columns else "file"
        files_by_cow = {c: g.sort_values(key)["file"].tolist() for c, g in meta.groupby("cow")}
        seen, rebuilt = {}, []
        for lab, p in zip(labels, paths):
            cow = label_names[lab]
            k = seen.get(cow, 0)
            seen[cow] = k + 1
            lst = files_by_cow.get(cow)
            rebuilt.append(f"{cow}/{lst[k]}" if lst and k < len(lst) else p)
        bad = [c for c in seen if c.lower() not in EXCLUDE_CLASSES and len(files_by_cow.get(c, [])) != seen[c]]
        paths = rebuilt
        print(f"File names rebuilt from muzzles.csv (assumes files within a cow are in sorted order). "
              f"Cows whose image count differs from the CSV: {len(bad)}")
    else:
        print("WARNING: loader paths do not contain the original file names and no metadata CSV is available; "
              "order_in_cow and source prefixes are unreliable.")


def parse_name(cow, path, i):
    fname = (path or "").replace("\\", "/").split("/")[-1] or f"{cow}_row{i:05d}.jpg"
    stem = fname.rsplit(".", 1)[0]
    rest = stem[len(cow) + 1:] if stem.startswith(cow + "_") else stem
    m = re.match(r"([A-Za-z]+)[_-]?(\d+)", rest)
    src = m.group(1).upper() if m else "OTHER"       # filename prefix, a proxy for camera/source
    num = int(m.group(2)) if m else -1               # frame number inside that prefix
    return fname, src, num


rows = []
for i, (lab, p) in enumerate(zip(labels, paths)):
    cow = label_names[lab]
    if cow.lower() in EXCLUDE_CLASSES:
        continue
    fname, src, num = parse_name(cow, p, i)
    rows.append(dict(ds_idx=i, image_id=f"{cow}/{fname}", cow=cow, file=fname, src=src, num=num))

man = pd.DataFrame(rows)
if meta is not None and {"cow", "file", "width", "height"} <= set(meta.columns):
    orig = meta[["cow", "file", "width", "height"]].rename(columns={"width": "orig_width", "height": "orig_height"})
    man = man.merge(orig.drop_duplicates(["cow", "file"]), on=["cow", "file"], how="left")
    print(f"Original sizes matched for {man.orig_width.notna().sum()} of {len(man)} images")
man = man.sort_values(["cow", "src", "num", "file"]).reset_index(drop=True)
man["order_in_cow"] = man.groupby("cow").cumcount()    # file order: a capture-order PROXY, not a timestamp
assert man.image_id.is_unique, "duplicate image IDs"
print(f"{len(man)} images, {man.cow.nunique()} cows (dropped {len(labels) - len(man)} rows from excluded classes)")
man.head()

## 2. Per-image features: size, quality flags and a perceptual hash

One pass over the pixels computes width/height, mean brightness, contrast, a blur score (variance of the Laplacian) and a 256-bit difference hash (dHash) used for the near-duplicate audit. Flags are **recorded, not used to drop images**: the brief wants failure cases and "recapture" decisions kept visible.

In [ ]:
def get_pil(ds_idx):
    return train[int(ds_idx)]["image"]


def analyse(pil):
    g = pil.convert("L")
    a = np.asarray(g, dtype=np.float32)
    lap = -4 * a[1:-1, 1:-1] + a[:-2, 1:-1] + a[2:, 1:-1] + a[1:-1, :-2] + a[1:-1, 2:]
    small = np.asarray(g.resize((HASH_SIZE + 1, HASH_SIZE), PILImage.LANCZOS), dtype=np.float32)
    bits = (small[:, 1:] > small[:, :-1]).flatten()
    return dict(width=pil.size[0], height=pil.size[1], luma_mean=float(a.mean()), luma_std=float(a.std()),
                blur_var=float(lap.var()), hash=np.packbits(bits))


feats = {}
for i in tqdm(man.ds_idx.tolist(), desc="analysing images"):
    feats[i] = analyse(get_pil(i))

for k in ["width", "height", "luma_mean", "luma_std", "blur_var"]:
    man[k] = [feats[i][k] for i in man.ds_idx]
H = np.stack([feats[i]["hash"] for i in man.ds_idx])          # (N, 32) uint8, row order == man order
man["dhash_hex"] = [h.tobytes().hex() for h in H]
H.shape

In [ ]:
blur_cut = man.blur_var.quantile(0.05)
man["flag_blurry"] = man.blur_var < blur_cut
man["flag_dark"] = man.luma_mean < 40
man["flag_bright"] = man.luma_mean > 215
man["flag_low_contrast"] = man.luma_std < 20
man["flag_small"] = man[["width", "height"]].min(axis=1) < MIN_SIDE
flag_cols = [c for c in man.columns if c.startswith("flag_")]
man["quality_flags"] = man[flag_cols].apply(lambda r: ";".join(c[5:] for c in flag_cols if r[c]), axis=1)
print("Flag counts (thresholds are heuristics: check a few flagged images by eye):")
print(man[flag_cols].sum().to_string())
if "orig_width" in man.columns and man.orig_width.notna().any():
    short = man[["orig_width", "orig_height"]].min(axis=1)
    print(f"\nOriginal short side (from muzzles.csv): median {short.median():.0f} px, 5th percentile {short.quantile(0.05):.0f} px "
          f"(this loaded copy: median {man[['width', 'height']].min(axis=1).median():.0f} px)")

## 3. Manifest overview

In [ ]:
def major(s):
    vc = s.value_counts()
    return sorted(vc.items(), key=lambda kv: (-kv[1], kv[0]))[0][0]


cow_info = man.groupby("cow").agg(n_images=("image_id", "size"), n_sources=("src", "nunique"), major_src=("src", major))
major_src_of = cow_info.major_src.to_dict()

print(cow_info.n_images.describe().round(1).to_string())
print("\nImages per source prefix:\n", man.src.value_counts().to_string())
print(f"\nCows with more than one source prefix: {(cow_info.n_sources > 1).sum()} of {len(cow_info)}")
print("Cows by majority source:\n", cow_info.major_src.value_counts().to_string())

fig, ax = plt.subplots(1, 3, figsize=(13, 3))
ax[0].hist(cow_info.n_images, bins=20); ax[0].set_title("images per cow")
man.src.value_counts().plot.bar(ax=ax[1]); ax[1].set_title("images per source prefix")
ax[2].hist(man[["width", "height"]].min(axis=1), bins=30); ax[2].set_title("short side (px)")
plt.tight_layout(); plt.show()

In [ ]:
# Is the file-number span of each cow small? A narrow span suggests one burst, i.e. a single capture session.
span = man[man.src == man.cow.map(major_src_of)].groupby("cow").num.agg(lambda s: s.max() - s.min())
print("Frame-number span inside each cow's majority source (smaller = tighter burst):")
print(span.describe().round(1).to_string())
print("\nCapture sessions are NOT recorded in this dataset: treat every experiment as single-session unless you can prove otherwise.")

## 4. Near-duplicate audit

`nn_same` is the Hamming distance from each image to its closest image of the **same cow**; `nn_other` is the distance to the closest image of a **different cow**. Pick `DUP_BITS` so that it sits well below the `nn_other` bulk. Different-cow pairs below the threshold are either label noise or a threshold that is too loose.

dHash catches exact and near-exact copies (re-saves, crops of one frame). It does **not** catch burst frames where the animal moved; those are handled by the guard-frame rule below.

In [ ]:
POP = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint16)


def hamming(a, b):
    a, b = np.atleast_2d(a), np.atleast_2d(b)
    return POP[np.bitwise_xor(a[:, None, :], b[None, :, :])].sum(-1)


def nearest_distances(H, cow_codes, thr, chunk=128):
    N = len(H)
    same = np.full(N, 999, dtype=np.int32)
    other = same.copy()
    close = []
    for s in tqdm(range(0, N, chunk), desc="hash distances"):
        e = min(N, s + chunk)
        d = hamming(H[s:e], H).astype(np.int32)
        d[np.arange(e - s), np.arange(s, e)] = 999
        same_mask = cow_codes[s:e, None] == cow_codes[None, :]
        same[s:e] = np.where(same_mask, d, 999).min(1)
        d_other = np.where(same_mask, 999, d)
        other[s:e] = d_other.min(1)
        for a, b in np.argwhere(d_other <= thr):
            if s + a < b:
                close.append((s + int(a), int(b), int(d_other[a, b])))
    return same, other, close


cow_codes = man.cow.astype("category").cat.codes.to_numpy()
nn_same, nn_other, close_pairs = nearest_distances(H, cow_codes, DUP_BITS)

plt.figure(figsize=(7, 3))
bins = np.arange(0, 132, 4)
plt.hist([nn_same[nn_same < 999], nn_other[nn_other < 999]], bins=bins, label=["nearest, same cow", "nearest, other cow"])
plt.axvline(DUP_BITS, color="k", ls="--", label=f"DUP_BITS = {DUP_BITS}")
plt.xlabel("dHash Hamming distance (bits of 256)"); plt.legend(); plt.tight_layout(); plt.show()

print(f"Images with a same-cow near-duplicate (<= {DUP_BITS} bits): {(nn_same <= DUP_BITS).sum()}")
print(f"Different-cow pairs at or below the threshold: {len(close_pairs)}  (should be ~0; investigate if not)")
for i, j, d in sorted(close_pairs, key=lambda t: t[2])[:10]:
    print(f"  d={d:3d}  {man.image_id[i]}  <->  {man.image_id[j]}")

In [ ]:
def dup_clusters(H, man, thr):
    parent = np.arange(len(man))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for _, g in man.groupby("cow"):
        idx = g.index.to_numpy()
        d = hamming(H[idx], H[idx])
        ii, jj = np.where(np.triu(d <= thr, k=1))
        for a, b in zip(idx[ii], idx[jj]):
            ra, rb = find(a), find(b)
            if ra != rb:
                parent[rb] = ra
    return np.array([find(i) for i in range(len(man))])


man["dup_cluster"] = dup_clusters(H, man, DUP_BITS)
man["in_dup_cluster"] = man.groupby("dup_cluster").dup_cluster.transform("size") > 1
print(f"{man.in_dup_cluster.sum()} images ({man.in_dup_cluster.mean():.1%}) sit in a same-cow near-duplicate cluster; "
      f"{man[man.in_dup_cluster].cow.nunique()} cows affected")

## 5. Identity-disjoint split (cows, not images)

Whole cows go to train, dev or test, stratified by each cow's majority source prefix so the camera mix is similar in every split. The dev cows are for choosing thresholds; the test cows stay frozen until the final evaluation.

In [ ]:
rng = np.random.default_rng(SEED)
split_of = {}
for src, grp in cow_info.groupby("major_src"):
    cows = sorted(grp.index.tolist())
    cows = [cows[i] for i in rng.permutation(len(cows))]
    n = len(cows)
    n_test = int(round(n * SPLIT_FRACTIONS["test"]))
    n_dev = int(round(n * SPLIT_FRACTIONS["dev"]))
    for c in cows[:n_test]:
        split_of[c] = "test"
    for c in cows[n_test:n_test + n_dev]:
        split_of[c] = "dev"
    for c in cows[n_test + n_dev:]:
        split_of[c] = "train"

man["split"] = man.cow.map(split_of)
summary = man.groupby("split").agg(cows=("cow", "nunique"), images=("image_id", "size"))
print(summary.to_string())
print("\nCows per split and majority source:")
print(pd.crosstab(cow_info.index.map(split_of), cow_info.major_src).to_string())
assert len({c for c, s in split_of.items() if s == "train"} & {c for c, s in split_of.items() if s != "train"}) == 0

## 6. Enrolment / query partitions for dev and test cows

Rules for each held-out cow:

- Use the cow's **majority source** only, so genuine pairs are not systematically cross-camera. Images from the other source become `held_other_source`, reserved for a separate cross-source test.
- **Enrolment** images are the first ones in file order that are not near-duplicates of each other (up to `N_ENROLL`).
- A **query** must be more than `GUARD_FRAMES` frames away from every enrolment frame and not a near-duplicate of one. Near-duplicate queries are collapsed to one.
- A cow with fewer than `MIN_QUERY` usable queries is marked ineligible and contributes no pairs.

In [ ]:
def build_roles(g, major_src, dup_bits, guard, n_enroll, min_query):
    g = g.sort_values("order_in_cow")
    roles = {i: "held_other_source" for i in g.index[g.src != major_src]}
    idx = list(g.index[g.src == major_src])
    pos = {i: k for k, i in enumerate(idx)}

    enroll = []
    for i in idx:
        if len(enroll) >= n_enroll:
            break
        if not enroll or hamming(H[i], H[enroll]).min() > dup_bits:
            enroll.append(i)

    query = []
    for i in idx:
        if i in enroll:
            continue
        if min(abs(pos[i] - pos[e]) for e in enroll) <= guard:
            roles[i] = "excluded_guard"
        elif hamming(H[i], H[enroll]).min() <= dup_bits:
            roles[i] = "excluded_near_dup_of_enroll"
        elif query and hamming(H[i], H[query]).min() <= dup_bits:
            roles[i] = "excluded_query_dup"
        else:
            query.append(i)
    for e in enroll:
        roles[e] = "enroll"
    for q in query:
        roles[q] = "query"
    return roles, (len(enroll) >= 1 and len(query) >= min_query)


man["role"] = np.where(man.split == "train", "train", "unassigned").astype(object)
cow_eligible = {}
for cow, g in man[man.split != "train"].groupby("cow"):
    roles, ok = build_roles(g, major_src_of[cow], DUP_BITS, GUARD_FRAMES, N_ENROLL, MIN_QUERY)
    for i, r in roles.items():
        man.at[i, "role"] = r
    cow_eligible[cow] = ok
man["cow_eligible"] = man.cow.map(cow_eligible).fillna(False).astype(bool)

print(pd.crosstab(man.split, man.role).to_string())
elig = pd.Series(cow_eligible)
print("\nEligible held-out cows:", {s: int(elig[[c for c in elig.index if split_of[c] == s]].sum()) for s in ["dev", "test"]},
      "of", {s: int(sum(split_of[c] == s for c in elig.index)) for s in ["dev", "test"]})
assert (man.role != "unassigned").all()

### Sensitivity of the rules to the near-duplicate threshold

In [ ]:
held = man[man.split != "train"]
rows = []
for thr in [0, 8, 16, 24, 32, 48]:
    ok_n, q_n, ex_n = 0, [], 0
    for cow, g in held.groupby("cow"):
        roles, ok = build_roles(g, major_src_of[cow], thr, GUARD_FRAMES, N_ENROLL, MIN_QUERY)
        ok_n += ok
        q_n.append(sum(r == "query" for r in roles.values()))
        ex_n += sum(r.startswith("excluded") for r in roles.values())
    rows.append(dict(dup_bits=thr, eligible_cows=ok_n, mean_queries_per_cow=round(float(np.mean(q_n)), 1), excluded_images=ex_n))
pd.DataFrame(rows)

## 7. Genuine and impostor pairs

- **genuine**: every (enrolment, query) combination of the same cow.
- **impostor**: a query against an enrolment image of a *different* cow with the **same majority source**, so camera differences cannot separate genuine from impostor.
- **impostor_cross_source**: impostors with a different majority source, kept as a separate diagnostic list. Do not mix them into the primary metrics.

Pairs share images, so they are not independent samples. Report confidence intervals with a bootstrap over **cows**, not over pairs.

In [ ]:
ids = man.image_id.to_numpy()
cow_arr = man.cow.to_numpy()
src_arr = man.src.to_numpy()


def make_pairs(split):
    rng = np.random.default_rng(SEED + {"dev": 1, "test": 2}[split])
    cows = sorted(c for c, ok in cow_eligible.items() if ok and split_of[c] == split)
    enr = {c: man.index[(man.cow == c) & (man.role == "enroll")].to_numpy() for c in cows}
    qry = {c: man.index[(man.cow == c) & (man.role == "query")].to_numpy() for c in cows}
    rows = []

    def add(e, q, label, ptype):
        rows.append(dict(split=split, enroll_id=ids[e], query_id=ids[q], cow_enroll=cow_arr[e], cow_query=cow_arr[q],
                         label=label, pair_type=ptype, src_enroll=src_arr[e], src_query=src_arr[q]))

    for c in cows:
        same_src = [o for o in cows if o != c and major_src_of[o] == major_src_of[c]]
        diff_src = [o for o in cows if o != c and major_src_of[o] != major_src_of[c]]
        for q in qry[c]:
            for e in enr[c]:
                add(e, q, 1, "genuine")
            for o in rng.permutation(same_src)[:IMPOSTORS_PER_QUERY]:
                add(rng.choice(enr[str(o)]), q, 0, "impostor")
            for o in rng.permutation(diff_src)[:CROSS_SOURCE_IMPOSTORS_PER_QUERY]:
                add(rng.choice(enr[str(o)]), q, 0, "impostor_cross_source")
    return pd.DataFrame(rows)


pairs = {s: make_pairs(s) for s in ["dev", "test"]}
for s, p in pairs.items():
    print(f"{s}: {len(p)} pairs")
    print(p.pair_type.value_counts().to_string(), "\n")

for s, p in pairs.items():
    n_imp = int((p.pair_type == "impostor").sum())
    if n_imp:
        print(f"{s}: {n_imp} impostor pairs -> with zero errors seen, the 95% upper bound on FAR is about {3 / n_imp:.2%} "
              f"(rule of three, and optimistic because pairs share images). Claim TAR at a FAR below that only with caution.")

## 8. Leakage checks (all must pass before freezing)

In [ ]:
split_cows = {s: set(man.cow[man.split == s]) for s in ["train", "dev", "test"]}
assert not (split_cows["train"] & split_cows["dev"]) and not (split_cows["train"] & split_cows["test"]) \
    and not (split_cows["dev"] & split_cows["test"]), "a cow appears in two splits"

role_of = man.set_index("image_id").role
split_by_id = man.set_index("image_id").split
idx_of = {v: i for i, v in enumerate(ids)}

for s, p in pairs.items():
    assert (role_of[p.enroll_id].to_numpy() == "enroll").all(), "pair uses a non-enrolment image as enrolment"
    assert (role_of[p.query_id].to_numpy() == "query").all(), "pair uses a non-query image as query"
    assert (split_by_id[p.enroll_id].to_numpy() == s).all() and (split_by_id[p.query_id].to_numpy() == s).all()
    assert ((p.cow_enroll == p.cow_query) == (p.label == 1)).all(), "label does not match cow identity"
    assert not set(p.enroll_id) & set(p.query_id), "an image is both enrolment and query"

viol = 0
for cow, ok in cow_eligible.items():
    if not ok:
        continue
    e = man.index[(man.cow == cow) & (man.role == "enroll")]
    q = man.index[(man.cow == cow) & (man.role == "query")]
    viol += int((hamming(H[e], H[q]) <= DUP_BITS).sum())
print("enrolment/query near-duplicate violations:", viol)
assert viol == 0

cross_split = [(i, j, d) for i, j, d in close_pairs if man.split[i] != man.split[j]]
print("different-cow near-duplicate pairs that straddle splits:", len(cross_split))
print("ALL CHECKS PASSED" if not cross_split else "CHECK the straddling pairs above: same picture filed under two cows?")

## 9. Visual check: roles on two test cows

In [ ]:
ROLE_COLORS = {"enroll": "tab:green", "query": "tab:blue", "excluded_guard": "tab:orange",
               "excluded_near_dup_of_enroll": "tab:red", "excluded_query_dup": "tab:red", "held_other_source": "tab:purple"}


def show_cow(cow, max_imgs=14):
    g = man[man.cow == cow].sort_values("order_in_cow").head(max_imgs)
    fig, axes = plt.subplots(1, len(g), figsize=(1.7 * len(g), 2.4))
    for ax, (_, r) in zip(np.atleast_1d(axes), g.iterrows()):
        ax.imshow(get_pil(r.ds_idx).convert("RGB")); ax.axis("off")
        ax.set_title(f"{r.role}\n{r.src}{r.num}", fontsize=6, color=ROLE_COLORS.get(r.role, "black"))
    fig.suptitle(cow, fontsize=9); plt.tight_layout(); plt.show()


for cow in [c for c, ok in cow_eligible.items() if ok and split_of[c] == "test"][:2]:
    show_cow(cow)

## 10. Freeze: save the manifest, the split and the pairs

In [ ]:
man.drop(columns=["dup_cluster"]).to_csv(OUT / "manifest.csv", index=False)
for s, p in pairs.items():
    p.to_csv(OUT / f"pairs_{s}.csv", index=False)

splits = dict(
    seed=SEED, config={k: (v if not isinstance(v, set) else sorted(v)) for k, v in CONFIG.items()},
    cows={s: sorted(split_cows[s]) for s in ["train", "dev", "test"]},
    eligible_cows={s: sorted(c for c, ok in cow_eligible.items() if ok and split_of[c] == s) for s in ["dev", "test"]},
    roles={k: v for k, v in man.set_index("image_id").role.items()},
)
blob = json.dumps(splits, sort_keys=True, indent=1).encode()
(OUT / "splits.json").write_bytes(blob)
digest = hashlib.sha256(blob).hexdigest()
(OUT / "splits.sha256").write_text(digest + "\n")
print("splits.json sha256:", digest)
print("Written:", ", ".join(sorted(p.name for p in OUT.iterdir())))

In [ ]:
n_imp = {s: int((p.pair_type == "impostor").sum()) for s, p in pairs.items()}
n_gen = {s: int((p.pair_type == "genuine").sum()) for s, p in pairs.items()}
protocol = f"""# Evaluation protocol (draft generated {pd.Timestamp.now():%Y-%m-%d})

**Dataset:** {CONFIG['dataset']}, derived from Xiong, Li and Erickson (2022), Zenodo 10.5281/zenodo.6324361, CC BY 4.0.
{len(man)} images of {man.cow.nunique()} animals. Capture sessions are not recorded, so every experiment is **single-session**.
Camera/source is only known through the filename prefix: {", ".join(f"{k} ({v})" for k, v in man.src.value_counts().items())}.

**Split (seed {SEED}, whole animals, stratified by majority source prefix):**
{chr(10).join(f"- {s}: {summary.loc[s, 'cows']} cows, {summary.loc[s, 'images']} images" for s in ['train', 'dev', 'test'])}

**Enrolment / query:** up to {N_ENROLL} enrolment images per held-out cow (non-duplicate, first in file order); queries are more than
{GUARD_FRAMES} frames from every enrolment frame and not near-duplicates (dHash <= {DUP_BITS}/256 bits). Only the majority-source images are used;
the rest are held back for a separate cross-source test.

**Pairs:** genuine = enrolment x query of the same cow; impostor = query vs enrolment of a different cow with the same majority source.
Dev: {n_gen['dev']} genuine / {n_imp['dev']} impostor. Test: {n_gen['test']} genuine / {n_imp['test']} impostor.
Cross-source impostors are reported separately.

**Rules:** thresholds (accept / reject / recapture) are chosen on dev only. The test split is frozen (sha256 {digest[:16]}...) and used once.
Artificial distortions are applied to query images only after this split is frozen. Confidence intervals come from a bootstrap over cows.
Metrics: ROC/AUROC, EER, TAR at a stated FAR (only where the impostor count supports it), extraction failure rate, time per image and per match.
"""
(OUT / "protocol.md").write_text(protocol)
print(protocol)

## Next steps

- Reload the frozen split anywhere with `json.load(open("outputs/splits.json"))` and `pd.read_csv("outputs/manifest.csv")`.
- Inspect the images flagged in `manifest.csv` (`quality_flags`) and the `held_other_source` images before annotating.
- Pick the 60-100 image pilot annotation set from **train** cows only, so annotation effort never touches the frozen test animals.
- Rerun this notebook with `LOCAL_DIR` pointing at the Zenodo originals before building the landmark extractor, then check that the cow IDs and split still make sense (file names may differ).